In [1]:
%cd /content
import os
if not os.path.exists("valley-outfitters-analysis"):
    !git clone https://github.com/LSRajan/valley-outfitters-analysis.git
%cd valley-outfitters-analysis
!git pull

import pandas as pd
pd.set_option("display.max_columns", None)

/content
Cloning into 'valley-outfitters-analysis'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 68 (delta 20), reused 57 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 3.91 MiB | 8.28 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/valley-outfitters-analysis
Already up to date.


#0. Setup

In [2]:
import duckdb
con = duckdb.connect()
for t in ["orders", "customers", "order_items", "products", "returns", "web_sessions"]:
    con.sql(f"CREATE TABLE {t} AS SELECT * FROM 'data/clean/{t}.csv'")
con.sql("SHOW TABLES")

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ customers    │
│ order_items  │
│ orders       │
│ products     │
│ returns      │
│ web_sessions │
└──────────────┘

In [3]:
con.query(
"""
SUMMARIZE orders
""")

┌─────────────────┬─────────────┬────────────┬────────────┬───────────────┬────────────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────┬─────────────────┐
│   column_name   │ column_type │    min     │    max     │ approx_unique │            avg             │        std         │        q25         │        q50         │        q75        │ count │ null_percentage │
│     varchar     │   varchar   │  varchar   │  varchar   │     int64     │          varchar           │      varchar       │      varchar       │      varchar       │      varchar      │ int64 │  decimal(9,2)   │
├─────────────────┼─────────────┼────────────┼────────────┼───────────────┼────────────────────────────┼────────────────────┼────────────────────┼────────────────────┼───────────────────┼───────┼─────────────────┤
│ order_id        │ VARCHAR     │ ORD-200000 │ ORD-224999 │         24966 │ NULL                       │ NULL               │ NULL              

In [4]:
con.query(
"""
SUMMARIZE customers
""")

┌──────────────────┬─────────────┬─────────────┬────────────┬───────────────┬────────────────────────────┬───────┬────────────┬────────────┬────────────┬───────┬─────────────────┐
│   column_name    │ column_type │     min     │    max     │ approx_unique │            avg             │  std  │    q25     │    q50     │    q75     │ count │ null_percentage │
│     varchar      │   varchar   │   varchar   │  varchar   │     int64     │          varchar           │ int32 │  varchar   │  varchar   │  varchar   │ int64 │  decimal(9,2)   │
├──────────────────┼─────────────┼─────────────┼────────────┼───────────────┼────────────────────────────┼───────┼────────────┼────────────┼────────────┼───────┼─────────────────┤
│ customer_id      │ VARCHAR     │ C1000       │ C6319      │          5634 │ NULL                       │  NULL │ NULL       │ NULL       │ NULL       │  5048 │            0.00 │
│ first_name       │ VARCHAR     │ aisha       │ zainab     │            40 │ NULL                  

In [5]:
con.query(
"""
SUMMARIZE products
""")

┌──────────────┬─────────────┬───────────────────────┬─────────────────────────┬───────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────┬───────┬─────────────────┐
│ column_name  │ column_type │          min          │           max           │ approx_unique │         avg         │        std         │        q25         │        q50         │    q75     │ count │ null_percentage │
│   varchar    │   varchar   │        varchar        │         varchar         │     int64     │       varchar       │      varchar       │      varchar       │      varchar       │  varchar   │ int64 │  decimal(9,2)   │
├──────────────┼─────────────┼───────────────────────┼─────────────────────────┼───────────────┼─────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────────┼───────┼─────────────────┤
│ product_id   │ VARCHAR     │ P101                  │ P263                    │           142 │ NULL               

In [6]:
con.query(
"""
SUMMARIZE order_items
""")

┌───────────────┬─────────────┬────────────┬────────────┬───────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬───────┬─────────────────┐
│  column_name  │ column_type │    min     │    max     │ approx_unique │        avg        │        std         │        q25         │        q50        │        q75        │ count │ null_percentage │
│    varchar    │   varchar   │  varchar   │  varchar   │     int64     │      varchar      │      varchar       │      varchar       │      varchar      │      varchar      │ int64 │  decimal(9,2)   │
├───────────────┼─────────────┼────────────┼────────────┼───────────────┼───────────────────┼────────────────────┼────────────────────┼───────────────────┼───────────────────┼───────┼─────────────────┤
│ order_item_id │ BIGINT      │ 500001     │ 551903     │         55594 │ 525952.0          │ 14983.249847746632 │ 512973             │ 525961            │ 538921            │ 51903 │         

In [7]:
con.query(
"""
SUMMARIZE returns
""")

┌───────────────┬─────────────┬──────────────┬──────────────┬───────────────┬────────────────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬───────┬─────────────────┐
│  column_name  │ column_type │     min      │     max      │ approx_unique │            avg             │        std        │        q25         │        q50        │        q75         │ count │ null_percentage │
│    varchar    │   varchar   │   varchar    │   varchar    │     int64     │          varchar           │      varchar      │      varchar       │      varchar      │      varchar       │ int64 │  decimal(9,2)   │
├───────────────┼─────────────┼──────────────┼──────────────┼───────────────┼────────────────────────────┼───────────────────┼────────────────────┼───────────────────┼────────────────────┼───────┼─────────────────┤
│ return_id     │ VARCHAR     │ R7001        │ R9799        │          3219 │ NULL                       │ NULL              │ NULL         

In [8]:
con.query(
"""
SUMMARIZE web_sessions
""")

┌──────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬──────────────────────────┬────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬───────┬─────────────────┐
│     column_name      │ column_type │         min         │         max         │ approx_unique │           avg            │        std         │            q25             │            q50             │            q75             │ count │ null_percentage │
│       varchar        │   varchar   │       varchar       │       varchar       │     int64     │         varchar          │      varchar       │          varchar           │          varchar           │          varchar           │ int64 │  decimal(9,2)   │
├──────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼──────────────────────────┼────────────────────┼────────────────────────────┼────────────────────────────┼────────────────

##0.1 Views

Revenue and Refunds

In [9]:
con.query(
"""
CREATE OR REPLACE VIEW revenue AS
SELECT
  o.order_id,
  MAX(o.order_total) AS order_total,
  MAX(o.order_date) AS order_date,
  COUNT(r.return_id) AS num_items_returned,
  COALESCE(SUM(r.refund_amount), 0) AS total_refunded
FROM orders o
LEFT JOIN returns r
  ON o.order_id = r.order_id
  AND r.resolution = 'refund'
WHERE o.status != 'cancelled'
AND o.order_date IS NOT NULL
GROUP BY o.order_id"""
)

Customer Spending

In [10]:
con.query("""
CREATE OR REPLACE VIEW customers_spending AS
SELECT
  c.customer_id,
  MAX(c.signup_date) AS signup_date,
  MAX(c.date_of_birth) AS date_of_birth,
  MAX(c.city) AS city,
  MAX(c.country) AS country,
  MAX(c.marketing_opt_in) AS marketing_opt_in,
  COUNT(o.order_id) AS num_orders,
  ROUND(SUM(o.order_total), 2) AS total_spent,
  MIN(o.order_date) AS first_order_date,
  MAX(o.order_date) AS last_order_date
FROM customers c
JOIN orders o
  ON c.customer_id = o.customer_id
  AND o.status != 'cancelled'
GROUP BY c.customer_id
""")


Profits by Order

In [11]:
con.query(
"""
CREATE OR REPLACE VIEW order_profit AS
SELECT
  o.order_id,
  MAX(o.order_date) AS order_date,
  MAX(order_total) AS order_total,
  ROUND(SUM(oi.quantity * (oi.unit_price - p.cost_price)), 2) AS profit
FROM orders o
JOIN order_items oi
  ON o.order_id = oi.order_id
  AND o.status != 'cancelled'
  AND o.order_date IS NOT NULL
JOIN products p
  ON oi.product_id = p.product_id
  AND p.cost_price IS NOT NULL
GROUP BY o.order_id
"""
)

#1. General Revenue

###1.1 How much have refunds impacted revenue?

In [12]:
con.query("""
SELECT
  ROUND(SUM(order_total), 2) AS total_revenue,
  ROUND(SUM(total_refunded), 2) AS total_refunded,
  ROUND(SUM(total_refunded) / SUM(order_total)*100, 2) AS pct_refunded
FROM revenue
""")

┌───────────────┬────────────────┬──────────────┐
│ total_revenue │ total_refunded │ pct_refunded │
│    double     │     double     │    double    │
├───────────────┼────────────────┼──────────────┤
│    6568230.54 │       147104.7 │         2.24 │
└───────────────┴────────────────┴──────────────┘

**Finding:** The company has generated approximately £6.7m in revenue of which  2% (2.26%) was refunded. Refunds have a small impact on revenue so gross revenue can be used in exploration.

###1.2 How have revenue and volume changed over time? Is revenue driven by number of orders or by spending per order?

In [13]:
con.query("""
SELECT
  YEAR(order_date) AS 'year',
  MONTH(order_date) AS 'month',
  COUNT(order_id) AS num_orders,
  ROUND(SUM(order_total), 2) AS revenue,
  ROUND(SUM(order_total) / COUNT(order_id), 2) AS avg_order_revenue,
FROM revenue
WHERE order_date IS NOT NULL
GROUP BY year, month
ORDER BY year, month
""").df()

,year,month,num_orders,revenue,avg_order_revenue
0,2024,1,74,18109.50,244.72
1,2024,2,66,18629.03,282.26
2,2024,3,133,34844.48,261.99
3,2024,4,147,39088.04,265.91
4,2024,5,165,48807.33,295.80
5,2024,6,247,82357.66,333.43
6,2024,7,298,79305.85,266.13
7,2024,8,243,67732.95,278.74
8,2024,9,328,92486.60,281.97
9,2024,10,304,85260.66,280.46


**Finding:** Since the average order cost has only increased gradually, the primary source of revenue gain has been from the increase in the number of orders, which itself appears to peak around the last few months of each year.

The number of orders has gradually increased since the beginning of the company in January 2024. It generally sees an increase throughout the year, which could be attributed to seasonal trends or growth in the company.

The average revenue per order saw an initial growth through 2024 with both dips and peaks in various month. Perhaps due to smaller sample sizes. From 2025 onwards, the average has stabilised with possible a small increase in the overall average.

Overall, the revenue in the most recent months (late 2026) is more than 10x larger than the revenue in the earliest months of 2024. Most of 2025 saw revenue of over 200,000 but saw peaks in winter months where a larger number of orders were made. Most months of 2026 have earned revenues of over 300,000 but the final months of the year are not yet recorded.

In [14]:
con.query("""
SELECT
  MONTH(order_date) AS 'month',
  COUNT(order_id) AS num_orders,
  ROUND(SUM(order_total), 2) AS revenue,
  ROUND(SUM(order_total) / COUNT(order_id), 2) AS avg_order_revenue,
FROM revenue
WHERE order_date IS NOT NULL
GROUP BY month
ORDER BY month
""")

┌───────┬────────────┬───────────┬───────────────────┐
│ month │ num_orders │  revenue  │ avg_order_revenue │
│ int64 │   int64    │  double   │      double       │
├───────┼────────────┼───────────┼───────────────────┤
│     1 │       1243 │ 356824.12 │            287.07 │
│     2 │       1164 │ 338740.45 │            291.01 │
│     3 │       1782 │  512444.7 │            287.57 │
│     4 │       1697 │ 500120.67 │            294.71 │
│     5 │       2057 │ 592146.22 │            287.87 │
│     6 │       2198 │ 658328.48 │            299.51 │
│     7 │       2459 │ 687847.46 │            279.73 │
│     8 │       2216 │ 630943.78 │            284.72 │
│     9 │       2639 │  781811.8 │            296.25 │
│    10 │       1134 │ 321593.01 │            283.59 │
│    11 │       2070 │ 592892.58 │            286.42 │
│    12 │       2033 │ 594537.27 │            292.44 │
├───────┴────────────┴───────────┴───────────────────┤
│ 12 rows                                  4 columns │
└─────────

**Finding:** Average order revenue remains stable across months, however earlier months see much less orders than other months. Later months are missing 2026 data so the revenue and order metrics are misleading. General trend suggests larger amounts of orders in later months.

As mentioned previously, its possible that this trend is due to growth in the company rather than seasonal trends.

###1.3 How have profits changed over time? Is any change in profit due to change in the number of orders or the average profit per order.

In [15]:
con.query(
"""
SELECT
  YEAR(order_date) AS 'year',
  MONTH(order_date) AS 'month',
  COUNT(order_id) AS num_orders,
  ROUND(SUM(order_total), 2) AS revenue,
  ROUND(SUM(order_total) / COUNT(order_id), 2) AS avg_order_revenue,
  ROUND(SUM(profit), 2) AS profit,
  ROUND(SUM(profit) / COUNT(order_id), 2) AS avg_order_profit,
FROM order_profit
GROUP BY year, month
ORDER BY year, month
"""
).df()

,year,month,num_orders,revenue,avg_order_revenue,profit,avg_order_profit
0,2024,1,73,18063.99,247.45,9120.97,124.94
1,2024,2,66,18629.03,282.26,9307.88,141.03
2,2024,3,131,34456.22,263.02,16716.43,127.61
3,2024,4,142,38400.36,270.43,20227.71,142.45
4,2024,5,165,48807.33,295.80,24570.68,148.91
5,2024,6,241,81400.74,337.76,42118.06,174.76
6,2024,7,291,78846.61,270.95,37260.75,128.04
7,2024,8,235,66566.10,283.26,33219.65,141.36
8,2024,9,323,89902.59,278.34,44396.12,137.45
9,2024,10,301,84799.12,281.72,43880.41,145.78


**Finding:** The average profit per order has remained relatively consistent between 140-150. At most the average has slightly increased across time. Otherwise, profit is driven by the number of orders as is closely tied to the overall revenue.

In [16]:
con.query(
"""
SELECT
  MONTH(order_date) AS 'month',
  COUNT(order_id) AS num_orders,
  ROUND(SUM(order_total), 2) AS revenue,
  ROUND(SUM(order_total) / COUNT(order_id), 2) AS avg_order_revenue,
  ROUND(SUM(profit), 2) AS profit,
  ROUND(SUM(profit) / COUNT(order_id), 2) AS avg_order_profit,
FROM order_profit
GROUP BY month
ORDER BY month
"""
).df()

,month,num_orders,revenue,avg_order_revenue,profit,avg_order_profit
0,1,1218,353190.47,289.98,176241.64,144.70
1,2,1146,335662.47,292.90,167313.90,146.00
2,3,1754,507554.94,289.37,252945.07,144.21
3,4,1668,495435.07,297.02,246071.32,147.52
4,5,2026,585975.48,289.23,289608.57,142.95
5,6,2161,649878.84,300.73,326671.75,151.17
6,7,2422,682332.19,281.72,344016.59,142.04
7,8,2178,624922.60,286.92,309279.92,142.00
8,9,2600,772348.94,297.06,382432.95,147.09
9,10,1120,319730.19,285.47,162869.28,145.42


**Finding:** As before, the average profit per order has remained relatively consistent between 140-150. There does not appear to be any seasonality to the profits.

The profit would remain consistent regardless of seasonal changes in revenue if the difference is in overall volume not in particular high profit items seeing a larger proportion of overall sales. This is the likey explanation for the consistency in profit.

#2. Customers

###2.1 What percentage are repeat customers? What impact does this have on revenue?

In [17]:
con.query("""
SELECT
 (CASE
  WHEN num_orders > 1 THEN 'repeat'
  ELSE'one-time' END) AS repeat_customer,
 COUNT(customer_id) AS num_customers,
 ROUND(COUNT(customer_id)/ SUM(COUNT(customer_id)) OVER() * 100, 2) AS percentage_of_customers,
 SUM(num_orders) AS num_orders,
 ROUND(SUM(num_orders)/ SUM(SUM(num_orders)) OVER () * 100, 1) AS percentage_of_orders,
 ROUND(SUM(num_orders) / num_customers, 2) AS avg_orders_per_customer,
 ROUND(SUM(total_spent), 2) AS total_revenue,
 ROUND(SUM(total_spent) / SUM(num_orders), 2) AS average_order_total,
 ROUND(SUM(total_spent) / num_customers, 2) AS revenue_per_customer,
 ROUND(SUM(total_spent) / SUM(SUM(total_spent)) OVER () * 100, 1) AS percentage_of_revenue
FROM customers_spending
GROUP BY repeat_customer
ORDER BY num_customers DESC
""")


┌─────────────────┬───────────────┬─────────────────────────┬────────────┬──────────────────────┬─────────────────────────┬───────────────┬─────────────────────┬──────────────────────┬───────────────────────┐
│ repeat_customer │ num_customers │ percentage_of_customers │ num_orders │ percentage_of_orders │ avg_orders_per_customer │ total_revenue │ average_order_total │ revenue_per_customer │ percentage_of_revenue │
│     varchar     │     int64     │         double          │   int128   │        double        │         double          │    double     │       double        │        double        │        double         │
├─────────────────┼───────────────┼─────────────────────────┼────────────┼──────────────────────┼─────────────────────────┼───────────────┼─────────────────────┼──────────────────────┼───────────────────────┤
│ repeat          │          3771 │                   81.66 │      21738 │                 96.2 │                    5.76 │     6296711.7 │              289.66 │   

**Finding:** Repeat customers make up 80% of customers so far but make up 96% of all revenue.
Repeat customers have slightly higher order totals on average, but order significantly more often (5.8 vs 1.0) on average.

Converting one-time buyers to repeat customers should be a priority, as this is where most of the revenue lies.

### 2.2 What impact does opting into marketing have on revenue? What drives change, an increase in orders? Higher spending?

In [94]:
con.query("""
SELECT
  marketing_opt_in,
  COUNT(customer_id) AS num_customers,
  ROUND(num_customers/SUM(COUNT(customer_id)) OVER () *100, 1) AS percent_total_customers,
  ROUND(SUM(total_spent), 2) AS total_revenue,
  SUM(num_orders) AS total_orders,
  ROUND(SUM(total_spent)/SUM(num_orders), 2) AS average_spend_per_order,
  ROUND(AVG(total_spent), 2) AS average_spent_per_customer,
  ROUND(AVG(num_orders), 1) AS average_num_orders,
  ROUND(SUM(total_spent)/SUM(SUM(total_spent)) OVER() *100, 1) AS percent_total_revenue,
  ROUND(SUM(num_orders)/SUM(SUM(num_orders)) OVER() *100, 1) AS percent_total_orders,
FROM customers_spending
GROUP BY marketing_opt_in
ORDER BY marketing_opt_in DESC
""").df()

,marketing_opt_in,num_customers,percent_total_customers,total_revenue,total_orders,average_spend_per_order,average_spent_per_customer,average_num_orders,percent_total_revenue,percent_total_orders
0,True,2254,48.8,3380720.72,11545.0,292.83,1502.54,5.1,51.7,51.1
1,False,1993,43.2,2664256.86,9278.0,287.16,1341.52,4.7,40.8,41.1
2,<NA>,371,8.0,492737.81,1762.0,279.65,1328.13,4.7,7.5,7.8


**Finding:** Customers who opt in spend and order more on average. They represent a slightly higher proportion of revenue and orders than their proportion of total customers. Both average spend per order and average number of orders contribute to the larger share of revenue from customers who opt in. However, this difference can be attributed more to the number of orders than average spend per order.

Around 49% (vs 43% customers who opted out) of customers have opted in to marketing, and these customers represent 52% of revenue (vs 41%). Customers who have opted in have spent over £150 more on average (£1502 vs £1341) and order more often (5.1 vs 4.7).

###2.3 Revenue Concentration. How much do different customer spending groups contribute to the overall revenue?

In [95]:
con.query(
"""
WITH percentile_x10_grouping AS(
  SELECT
    * ,
    NTILE(10) OVER (ORDER BY total_spent DESC) AS percentile_group_x10
  FROM customers_spending
  WHERE total_spent IS NOT NULL
)
SELECT
  percentile_group_x10,
  SUM(num_orders) AS num_orders,
  ROUND(SUM(total_spent), 2) AS total_revenue,
  ROUND(SUM(num_orders)/SUM(SUM(num_orders)) OVER()*100, 1) AS percent_total_orders,
  ROUND(SUM(SUM(num_orders)) OVER (ORDER BY percentile_group_x10) / SUM(SUM(num_orders)) OVER () * 100, 1) AS cumulative_percent_orders,
  ROUND(SUM(total_spent)/SUM(SUM(total_spent)) OVER()*100, 1) AS percent_total_revenue,
  ROUND(SUM(SUM(total_spent)) OVER (ORDER BY percentile_group_x10) / SUM(SUM(total_spent)) OVER () * 100, 1) AS cumulative_percent_revenue
FROM percentile_x10_grouping
GROUP BY percentile_group_x10
ORDER BY percentile_group_x10
"""
)

┌──────────────────────┬────────────┬───────────────┬──────────────────────┬───────────────────────────┬───────────────────────┬────────────────────────────┐
│ percentile_group_x10 │ num_orders │ total_revenue │ percent_total_orders │ cumulative_percent_orders │ percent_total_revenue │ cumulative_percent_revenue │
│        int64         │   int128   │    double     │        double        │          double           │        double         │           double           │
├──────────────────────┼────────────┼───────────────┼──────────────────────┼───────────────────────────┼───────────────────────┼────────────────────────────┤
│                    1 │       9146 │    2849397.43 │                 40.5 │                      40.5 │                  43.6 │                       43.6 │
│                    2 │       2895 │     993587.31 │                 12.8 │                      53.3 │                  15.2 │                       58.8 │
│                    3 │       2225 │     721565.38 

**Finding:** The top 10% of customers contribute 44% of total revenue and 40% of orders, accounting for £2.8m in sales and over 9000 orders. The top 30% account for 63% of orders and nearly 70% of the total revenue. The greater percentage of revenue than orders implies that these groups spend more than other groups. However, their majority share of the revenue is primarily due to the volume of orders, as seen previously.

On the other hand, the bottom 50% of customers account for only 21% of orders and less than 15% of overall revenue. These customers spend less per order and order much less.

High impact customers should be a focus of the company. Retaining these customers is key for revenue.

#3. Products

### 3.1 How do different product categories perform? Does number of orders or spend drive performance?

In [19]:
con.query(
"""
SELECT
  p.category,
  COUNT(DISTINCT oi.order_id) AS num_orders,
  ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent,
  ROUND(SUM(oi.quantity * (oi.unit_price - p.cost_price)), 2) AS total_profit,
  ROUND(total_profit/total_spent*100, 2) AS margin,
  ROUND(total_spent/SUM(SUM(oi.quantity * oi.unit_price)) OVER ()*100, 1) AS percentage_overall_revenue,
  ROUND(total_profit/SUM(SUM(oi.quantity * (oi.unit_price - p.cost_price))) OVER ()*100, 1) AS percentage_overall_profit,
  ROUND(total_spent/num_orders, 2) AS average_category_spend,
  ROUND(total_profit/num_orders, 2) AS average_order_profit
FROM order_items oi
JOIN products p
  ON p.product_id = oi.product_id
  AND p.cost_price IS NOT NULL
JOIN orders o
  ON o.order_id = oi.order_id
  AND o.status != 'cancelled'
GROUP BY p.category
ORDER BY total_spent DESC
"""
)

┌─────────────┬────────────┬─────────────┬──────────────┬────────┬────────────────────────────┬───────────────────────────┬────────────────────────┬──────────────────────┐
│  category   │ num_orders │ total_spent │ total_profit │ margin │ percentage_overall_revenue │ percentage_overall_profit │ average_category_spend │ average_order_profit │
│   varchar   │   int64    │   double    │    double    │ double │           double           │          double           │         double         │        double        │
├─────────────┼────────────┼─────────────┼──────────────┼────────┼────────────────────────────┼───────────────────────────┼────────────────────────┼──────────────────────┤
│ camping     │       7832 │  2056780.71 │   1102396.99 │   53.6 │                       31.2 │                      33.2 │                 262.61 │               140.76 │
│ outerwear   │       7316 │  1777167.13 │    888135.65 │  49.97 │                       27.0 │                      26.7 │                 

**Finding:** The margin for each category remains similar at around 50%, so the difference in average order profit is due to the pricing of the item not necessarily due to profitability.

While most categories have somewhat similar number of orders (around 7000 excluding backpacks), most of the revenue is tied to only a few categories. Camping and Outerwear combined represent almost 60% (58.2%) of revenue and almost 60% (59.9%) of profits. Footwear and backpacks combine for just less than 30% of revenue and profits. Whereas base layers and accessories only see around £400k in revenue and £200k in profit, accounting for less than 15% of both revenue and profit.




### 3.2 How do different products perform? Does number of orders or spend drive performance?

In [20]:
con.query(
"""
SELECT
  p.product_id, p.product_name,
  COUNT(DISTINCT oi.order_id) AS num_orders,
  ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue,
  ROUND(SUM(oi.quantity * (oi.unit_price - p.cost_price)), 2) AS total_profit,
  ROUND(total_profit/total_revenue*100, 2) AS margin,
  ROUND(total_revenue/SUM(SUM(oi.quantity * oi.unit_price)) OVER ()*100, 1) AS percentage_overall_revenue,
  ROUND(total_profit/SUM(SUM(oi.quantity * (oi.unit_price - p.cost_price))) OVER ()*100, 1) AS percentage_overall_profit,
  ROUND(total_revenue/num_orders, 2) AS average_product_revenue,
  ROUND(total_profit/num_orders, 2) AS average_purchase_profit
FROM order_items oi
JOIN products p
  ON p.product_id = oi.product_id
  AND p.cost_price IS NOT NULL
JOIN orders o
  ON o.order_id = oi.order_id
  AND o.status != 'cancelled'
GROUP BY p.product_id, p.product_name
ORDER BY total_revenue DESC
"""
)

┌────────────┬──────────────────────────┬────────────┬───────────────┬──────────────┬────────┬────────────────────────────┬───────────────────────────┬─────────────────────────┬─────────────────────────┐
│ product_id │       product_name       │ num_orders │ total_revenue │ total_profit │ margin │ percentage_overall_revenue │ percentage_overall_profit │ average_product_revenue │ average_purchase_profit │
│  varchar   │         varchar          │   int64    │    double     │    double    │ double │           double           │          double           │         double          │         double          │
├────────────┼──────────────────────────┼────────────┼───────────────┼──────────────┼────────┼────────────────────────────┼───────────────────────────┼─────────────────────────┼─────────────────────────┤
│ P174       │ alpine sleeping mat      │        302 │      125783.0 │      56278.5 │  44.74 │                        1.9 │                       1.7 │                   416.5 │       

**Finding:** Products on both end of the spectrum in terms of total revenue both see a similar number of orders. The more expensive products see similar demand so see a larger proportion of the total revenue.

 Profit margins are more variable than categories, with values around generally between 40-60. However products on both ends see similar variation, so again the difference is likely primarily due to pricing and not profitability or demand.

#4. Revenue Losses

###4.1 How have refunds changed over time? Has their impact on revenue changed?

In [21]:
con.query("""
SELECT
  MONTH(order_date) AS 'month',
  COUNT(order_id) AS num_orders,
  ROUND(SUM(order_total), 2) AS revenue,
  ROUND(SUM(total_refunded)) AS total_refuned,
  ROUND(SUM(total_refunded)/SUM(order_total) * 100, 1) AS percent_revenue_refunded,
  ROUND(SUM(CASE WHEN total_refunded > 0 THEN 1 ELSE 0 END)/COUNT(order_id) * 100, 1) AS percent_orders_with_refunded
FROM revenue
WHERE order_date IS NOT NULL
GROUP BY month
ORDER BY month
""")

┌───────┬────────────┬───────────┬───────────────┬──────────────────────────┬──────────────────────────────┐
│ month │ num_orders │  revenue  │ total_refuned │ percent_revenue_refunded │ percent_orders_with_refunded │
│ int64 │   int64    │  double   │    double     │          double          │            double            │
├───────┼────────────┼───────────┼───────────────┼──────────────────────────┼──────────────────────────────┤
│     1 │       1243 │ 356824.12 │        8328.0 │                      2.3 │                          6.7 │
│     2 │       1164 │ 338740.45 │        7873.0 │                      2.3 │                          7.8 │
│     3 │       1782 │  512444.7 │       10943.0 │                      2.1 │                          6.5 │
│     4 │       1697 │ 500120.67 │       12506.0 │                      2.5 │                          7.3 │
│     5 │       2057 │ 592146.22 │       11582.0 │                      2.0 │                          6.5 │
│     6 │       219

**Finding:** Refunds are stable across months, with consistent percentage of revenue being refunded and consistent amount of orders with refunds present.

### 4.2 What were the reasons for returns? What impact did each reason have on revenue

In [22]:
con.query("""
SELECT
  COALESCE(reason, 'unknown') AS reason,
  COUNT(return_id) AS num_returns,
  ROUND(SUM(refund_amount), 2) AS total_return_amount,
  ROUND(AVG(refund_amount), 2) AS average_refund_amount,
  ROUND(COUNT(return_id) / SUM(COUNT(return_id)) OVER() *100, 1) AS percent_of_returns,
  ROUND(SUM(refund_amount) / SUM(SUM(refund_amount)) OVER() *100, 1) AS percent_total_return_amount
FROM returns
GROUP BY reason
ORDER BY total_return_amount DESC
""")

┌─────────────────┬─────────────┬─────────────────────┬───────────────────────┬────────────────────┬─────────────────────────────┐
│     reason      │ num_returns │ total_return_amount │ average_refund_amount │ percent_of_returns │ percent_total_return_amount │
│     varchar     │    int64    │       double        │        double         │       double       │           double            │
├─────────────────┼─────────────┼─────────────────────┼───────────────────────┼────────────────────┼─────────────────────────────┤
│ size            │         972 │            83644.87 │                 87.77 │               34.7 │                        34.2 │
│ faulty product  │         497 │            44124.32 │                  89.5 │               17.8 │                        18.0 │
│ changed mind    │         334 │            29507.19 │                 90.51 │               11.9 │                        12.1 │
│ faulty delivery │         315 │             27399.6 │                 88.39 │    

**Finding:** The most common reason for returns is size, representing over 34% of returns and over 34% of the amount refunded. The next most common reason is a faulty product which represents around 18% of returns and 18% of the amount refunded. Around 11% of returns and amount returned is due to faulty delivery.

No particular reason results in a disproportionate amount refunded. The amount of revenue refunded by any given reason is due to the frequency of that reason.

### 4.3 What were the resolutions for returns? What impact did each have on revenue?

In [23]:
con.query("""
SELECT
  resolution,
  COUNT(return_id) AS num_returns,
  ROUND(SUM(refund_amount), 2) AS total_return_amount,
  ROUND(AVG(refund_amount), 2) AS average_refund_amount,
  ROUND(COUNT(return_id) / SUM(COUNT(return_id)) OVER() *100, 1) AS percent_of_returns,
  ROUND(SUM(refund_amount) / SUM(SUM(refund_amount)) OVER() *100, 1) AS percent_total_return_amount
FROM returns
GROUP BY resolution
ORDER BY total_return_amount DESC
""")

┌──────────────┬─────────────┬─────────────────────┬───────────────────────┬────────────────────┬─────────────────────────────┐
│  resolution  │ num_returns │ total_return_amount │ average_refund_amount │ percent_of_returns │ percent_total_return_amount │
│   varchar    │    int64    │       double        │        double         │       double       │           double            │
├──────────────┼─────────────┼─────────────────────┼───────────────────────┼────────────────────┼─────────────────────────────┤
│ refund       │        1694 │           151655.14 │                 91.14 │               60.5 │                        62.0 │
│ store credit │         568 │            47057.73 │                 84.64 │               20.3 │                        19.2 │
│ exchange     │         537 │             45764.8 │                 87.34 │               19.2 │                        18.7 │
└──────────────┴─────────────┴─────────────────────┴───────────────────────┴────────────────────┴───────

**Finding:** Around 60% (1694) of returns were resolved by refunding the customer, while 40% (1105) were either exchanged for store credit or item exchange. The average refund amount for store credit or exchanges (£84 and £87) were lower than refunds (£91), and directly redirect the money back into the company.

A possible suggestion would be to increase the number of returns which are resolved either by store credit or exchanges. Both size and faulty product are reasons for which exchanges are direct solutions rather than refunds.

How do return rates differ across product categories?

In [59]:
con.query(
"""
SELECT
  p.category,
  COUNT(oi.order_item_id) AS num_items,
  COUNT(r.return_id) AS num_returns,
  ROUND(COUNT(r.return_id) / COUNT(oi.order_item_id)*100, 1) AS return_percent,
  ROUND(SUM(r.refund_amount), 2) AS return_amount,
  ROUND(COUNT(oi.order_item_id) / SUM(COUNT(oi.order_item_id)) OVER()*100, 2) AS percent_total_items,
  ROUND(SUM(r.refund_amount) / SUM(SUM(r.refund_amount)) OVER()*100, 2) AS percent_total_return_amount
FROM order_items oi
JOIN products p
  ON p.product_id = oi.product_id
LEFT JOIN returns r
  ON r.order_item_id = oi.order_item_id
JOIN orders o
  ON oi.order_id = o.order_id
  AND o.status != 'cancelled'
GROUP BY p.category
"""
)

┌─────────────┬───────────┬─────────────┬────────────────┬───────────────┬─────────────────────┬─────────────────────────────┐
│  category   │ num_items │ num_returns │ return_percent │ return_amount │ percent_total_items │ percent_total_return_amount │
│   varchar   │   int64   │    int64    │     double     │    double     │       double        │           double            │
├─────────────┼───────────┼─────────────┼────────────────┼───────────────┼─────────────────────┼─────────────────────────────┤
│ footwear    │      7489 │         456 │            6.1 │      42456.02 │               15.57 │                       17.41 │
│ base layers │      7962 │         484 │            6.1 │      17859.96 │               16.55 │                        7.32 │
│ backpacks   │      5521 │         287 │            5.2 │      25036.79 │               11.48 │                       10.27 │
│ outerwear   │      8503 │         482 │            5.7 │      62536.07 │               17.68 │               

**Finding:** Return rates are very similar across categories, ranging between 5.2% - 6.1%. There are no concerns of a particular category being  resposible for a disproportionate number of returns. The difference in the total returned for each category is primarily due to the number of orders and pricing of the items.

# Web Activity

###5.1 Do customer conversion rates differ across devices?

In [58]:
con.query(
"""
SELECT
  device,
  COUNT(session_id) AS num_sessions,
  ROUND(SUM(converted)/COUNT(converted)*100,1) AS converted_percent,
  ROUND(COUNT(session_id) / SUM(COUNT(session_id)) OVER ()*100, 2) AS percent_all_sessions
FROM web_sessions
GROUP BY device
"""
)

┌─────────┬──────────────┬────────────────┬──────────────────────┐
│ device  │ num_sessions │ converted_rate │ percent_all_sessions │
│ varchar │    int64     │     double     │        double        │
├─────────┼──────────────┼────────────────┼──────────────────────┤
│ mobile  │        35993 │            3.7 │                59.99 │
│ tablet  │         4204 │            3.5 │                 7.01 │
│ desktop │        19803 │            3.3 │                33.01 │
└─────────┴──────────────┴────────────────┴──────────────────────┘

**Finding:** Conversion rates are similar across devices, with rates between 3.3%-3.7%. 60% of sessions are from mobile devices, of which 3.7% are converted to a sale.

###5.2 Are customer conversion rates similar across the traffic source?

In [63]:
con.query(
"""
SELECT
  traffic_source,
  COUNT(session_id) AS num_sessions,
  ROUND(COUNT(session_id) / SUM(COUNT(session_id)) OVER ()*100, 2) AS percent_total_sessions,
  ROUND(SUM(converted)/COUNT(converted) * 100,1) AS converted_percent
FROM web_sessions
GROUP BY traffic_source
"""
)

┌────────────────┬──────────────┬────────────────────────┬───────────────────┐
│ traffic_source │ num_sessions │ percent_total_sessions │ converted_percent │
│    varchar     │    int64     │         double         │      double       │
├────────────────┼──────────────┼────────────────────────┼───────────────────┤
│ ads            │         4923 │                   8.21 │               2.3 │
│ direct         │         7228 │                  12.05 │               4.9 │
│ organic        │        20998 │                   35.0 │               3.0 │
│ paid search    │         8206 │                  13.68 │               2.9 │
│ email          │         7207 │                  12.01 │               7.8 │
│ social         │         8467 │                  14.11 │               1.5 │
│ referral       │         2971 │                   4.95 │               3.2 │
└────────────────┴──────────────┴────────────────────────┴───────────────────┘

**Finding:** Email ads have the highest conversion with almost 8% conversion rate. Second is direct traffic with 4.9%. The most common source of traffic is organic representing 35% off traffic with 3% conversion. Regular ads and paid search ads both underperform this benchmark and have less than 3% conversion.

It is important to consider that this measures conversion for the instance the user visits the website. It does not consider following visits which may result in higher long-term conversion.

Paid searches appear when a customer is searching for the item itself, so the company should expect this to be higher than regular organic searches. Ads should also be a concern and should be higher conversion than organic. Although both may have a wider impact than first time visit conversion.

Email and direct have the highest conversion. This is to be expected given we have already found that marketing-opt-in customers spend and order more. Also, if a customer is visiting the site directly, it is presumably with greater intent to purchase an item than regular browsing.

###5.3 Do signed-in customers convert more often than otherwise?

In [66]:
con.query("""
SELECT
  CASE WHEN customer_id IS NULL THEN 'False' ELSE 'True' END AS signed_in,
  COUNT(session_id) AS num_sessions,
  ROUND(COUNT(session_id) / SUM(COUNT(session_id)) OVER ()*100, 1) AS percent_total_sessions,
  ROUND(SUM(converted)/COUNT(converted) * 100,1) AS converted_percent
FROM web_sessions
GROUP BY signed_in
ORDER BY signed_in DESC
"""
)

┌───────────┬──────────────┬────────────────────────┬───────────────────┐
│ signed_in │ num_sessions │ percent_total_sessions │ converted_percent │
│  varchar  │    int64     │         double         │      double       │
├───────────┼──────────────┼────────────────────────┼───────────────────┤
│ True      │        24264 │                   40.4 │               3.6 │
│ False     │        35736 │                   59.6 │               3.5 │
└───────────┴──────────────┴────────────────────────┴───────────────────┘

**Finding:** Signed-in customers convert only 0.1% more often than otherwise. Around 40% of sessions are from customers who have signed in. In general, signing in does not appear to have any impact on conversion.